In [ ]:
# Custom ResNet50 for Person Re-Identification - Built Directly in Notebook
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

class HorizontalMaxPool2d(nn.Module):
    """Custom horizontal max pooling for AlignedReID"""
    def __init__(self):
        super(HorizontalMaxPool2d, self).__init__()

    def forward(self, x):
        inp_size = x.size()
        return F.max_pool2d(input=x, kernel_size=(1, inp_size[3]))

class CustomResNet50ReID(nn.Module):
    """
    Custom ResNet50 built directly for Person Re-Identification
    No need to import from models module!
    """
    def __init__(self, num_classes=751, loss={'softmax'}, aligned=False, **kwargs):
        super(CustomResNet50ReID, self).__init__()
        self.loss = loss
        
        # Load pretrained ResNet50 backbone
        resnet50 = torchvision.models.resnet50(weights='ResNet50_Weights.DEFAULT')
        self.base = nn.Sequential(*list(resnet50.children())[:-2])  # Remove final avgpool and fc
        
        # Custom components for Re-ID
        self.class ifier = nn.Linear(2048, num_classes)
        self.feat_dim = 2048
        self.aligned = aligned
        self.horizon_pool = HorizontalMaxPool2d()
        
        # AlignedReID components (optional)
        if self.aligned:
            self.bn = nn.BatchNorm2d(2048)
            self.relu = nn.ReLU(inplace=True)
            self.conv1 = nn.Conv2d(2048, 128, kernel_size=1, stride=1, padding=0, bias=True)

    def forward(self, x):
        # Extract features through ResNet backbone
        x = self.base(x)
        
        # Local features for inference
        if not self.training:
            lf = self.horizon_pool(x)
            
        # AlignedReID processing during training
        if self.aligned and self.training:
            lf = self.bn(x)
            lf = self.relu(lf)
            lf = self.horizon_pool(lf)
            lf = self.conv1(lf)
            
        # Normalize local features
        if self.aligned or not self.training:
            lf = lf.view(lf.size()[0:3])
            lf = lf / torch.pow(lf, 2).sum(dim=1, keepdim=True).clamp(min=1e-12).sqrt()
        
        # Global features
        x = F.avg_pool2d(x, x.size()[2:])
        f = x.view(x.size(0), -1)
        
        # Return different outputs based on mode
        if not self.training:
            return f, lf  # Global and local features for inference
            
        # Training mode - get classification logits
        y = self.classifier(f)
        
        # Return based on loss type
        if self.loss == {'softmax'}:
            return y
        elif self.loss == {'metric'}:
            return (f, lf) if self.aligned else f
        elif self.loss == {'softmax', 'metric'}:
            return (y, f, lf) if self.aligned else (y, f)
        else:
            raise KeyError(f"Unsupported loss: {self.loss}")

# Example usage:
def create_reid_model(num_classes=751, aligned=True):
    """Factory function to create the Re-ID model"""
    model = CustomResNet50ReID(
        num_classes=num_classes,
        loss={'metric'},  # For inference/feature extraction
        aligned=aligned
    )
    model.eval()  # Set to evaluation mode
    return model

print("✅ Custom ResNet50 for Re-ID built directly in notebook!")
print("Usage: model = create_reid_model(num_classes=751, aligned=True)")

In [ ]:
# Practical Example: Using the Custom ResNet50 for Feature Extraction

def extract_person_features(model, person_image_tensor):
    """
    Extract Re-ID features from a person image
    
    Args:
        model: CustomResNet50ReID model
        person_image_tensor: Preprocessed person image tensor [1, 3, H, W]
    
    Returns:
        global_features: Global feature vector [1, 2048]
        local_features: Local feature vector [1, 128, H] (if aligned=True)
    """
    model.eval()
    with torch.no_grad():
        if model.aligned:
            global_features, local_features = model(person_image_tensor)
            return global_features, local_features
        else:
            features = model(person_image_tensor)
            return features, None

def compare_person_features(feat1, feat2):
    """
    Compare two person feature vectors using cosine similarity
    
    Returns:
        similarity_score: Float between 0-1 (higher = more similar)
    """
    # Normalize features
    feat1_norm = F.normalize(feat1, p=2, dim=1)
    feat2_norm = F.normalize(feat2, p=2, dim=1)
    
    # Cosine similarity
    similarity = torch.mm(feat1_norm, feat2_norm.t()).item()
    return similarity

# Example initialization and usage:
print("🚀 Ready to use custom ResNet50 for Re-ID!")
print("\n📋 Quick start:")
print("1. model = create_reid_model(num_classes=751, aligned=True)")
print("2. features = extract_person_features(model, person_tensor)")
print("3. similarity = compare_person_features(feat1, feat2)")

# Load pre-trained weights (when available)
def load_pretrained_weights(model, weights_path):
    """Load pre-trained Re-ID weights"""
    try:
        checkpoint = torch.load(weights_path, map_location='cpu')
        model.load_state_dict(checkpoint)
        print(f"✅ Loaded weights from {weights_path}")
    except FileNotFoundError:
        print(f"⚠️ Weights file not found: {weights_path}")
        print("Using ImageNet pre-trained backbone only")
    return model

In [ ]:
# Install required packages for Colab
!pip install scipy filterpy ultralytics torchvision transformers opencv-python scikit-learn

# Additional installs for ReID model
!pip install yacs tensorboard future

In [ ]:
!pip install scipy filterpy ultralytics torchvision 

In [2]:
import base64
import cv2
import torch
import numpy as np
import time
import sys
import logging
from collections import defaultdict, deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any
import json
from PIL import Image
from torchvision import transforms

ModuleNotFoundError: No module named 'cv2'

### Yolo

In [ ]:
from ultralytics import YOLO
model=YOLO('best.pt')  # load a finetuned YOLOv11n model

### Adding Kalman

In [ ]:
try:
    from filterpy.kalman import KalmanFilter
    from scipy.optimize import linear_sum_assignment
    ADVANCED_TRACKING_AVAILABLE = True
    print("✅ Kalman Filter and Hungarian Algorithm available")
except ImportError:
    ADVANCED_TRACKING_AVAILABLE = False
    print("⚠️ Advanced tracking not available. Install: pip install filterpy scipy")


In [ ]:
# Import necessary modules for ReID and CLIP
import os
from transformers import CLIPProcessor, CLIPModel
from sklearn.preprocessing import normalize
from scipy.spatial.distance import euclidean

# Check if we need to download ReID model checkpoint
checkpoint_path = "./log/checkpoint_ep300.pth.tar"
if not os.path.exists(checkpoint_path):
    print("⚠️ ReID checkpoint not found. Using simplified ReID for demo.")
    USE_REID = False
else:
    USE_REID = True

# Initialize logging for better debugging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✅ ReID and CLIP modules ready")

In [ ]:
class Aligned_Reid_class:
    """ReID class compatible with original AlignedReID ResNet50 checkpoint"""
    
    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.use_gpu = torch.cuda.is_available()
        print(f"ReID using device: {self.device}")
        
        # Check if original checkpoint exists
        checkpoint_path = "./log/checkpoint_ep300.pth.tar"
        if os.path.exists(checkpoint_path):
            print("✅ Found original AlignedReID checkpoint - loading full model")
            self.feature_extractor = self._init_aligned_reid_model(checkpoint_path)
            self.use_original_model = True
        else:
            print("⚠️ Original checkpoint not found - using ResNet50 fallback")
            self.feature_extractor = self._init_resnet50_fallback()
            self.use_original_model = False
        
        # Image preprocessing (same as original)
        self.img_transform = transforms.Compose([
            transforms.Resize((256, 128)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
    def _init_aligned_reid_model(self, checkpoint_path):
        """Initialize the original AlignedReID model with checkpoint"""
        try:
            # Try to import the models module (if available)
            import models
            from util.FeatureExtractor import FeatureExtractor
            
            # Initialize model exactly like the original
            model = models.init_model(
                name='resnet50', 
                num_classes=751, 
                loss={'softmax', 'metric'}, 
                use_gpu=self.use_gpu,
                aligned=True
            )
            
            # Load checkpoint
            if self.use_gpu:
                checkpoint = torch.load(checkpoint_path, map_location=lambda storage, loc: storage.cuda())
            else:
                checkpoint = torch.load(checkpoint_path, map_location='cpu')
                
            model.load_state_dict(checkpoint['state_dict'])
            model.eval()
            
            if self.use_gpu:
                model = model.cuda()
            
            # Use FeatureExtractor with layer '7' (same as original)
            feature_extractor = FeatureExtractor(model, ['7'])
            print("✅ Original AlignedReID model loaded successfully")
            return feature_extractor
            
        except ImportError as e:
            print(f"⚠️ Missing original modules: {e}")
            print("Falling back to ResNet50...")
            return self._init_resnet50_fallback()
        except Exception as e:
            print(f"⚠️ Error loading original model: {e}")
            print("Falling back to ResNet50...")
            return self._init_resnet50_fallback()
        
    def _init_resnet50_fallback(self):
        """Initialize ResNet50 fallback feature extractor"""
        from torchvision import models
        
        # Use pretrained ResNet50 as feature extractor (not ResNet18!)
        model = models.resnet50(pretrained=True)
        # Remove the final classification layer
        model = torch.nn.Sequential(*list(model.children())[:-1])
        model.eval()
        
        if self.use_gpu:
            model = model.cuda()
            
        print("✅ ResNet50 fallback model loaded")
        return model
    
    def pool2d(self, tensor, type='max'):
        """Pool2D function similar to original implementation"""
        tensor = tensor.cuda() if self.use_gpu else tensor
        sz = tensor.size()
        
        if type == 'max':
            maxpool = torch.nn.MaxPool2d(kernel_size=(sz[2] // 8, sz[3]))
            if self.use_gpu:
                maxpool = maxpool.cuda()
            x = maxpool(tensor)
        elif type == 'mean':
            x = torch.nn.functional.avg_pool2d(tensor, kernel_size=(sz[2] // 8, sz[3]))
        
        # Process similar to original
        res = [data.permute(2, 1, 0)[0] for data in x]
        return res
    
    def get_features(self, image_pil):
        """Extract features from PIL image"""
        try:
            # Convert PIL to tensor using the same transform as original
            img_tensor = self.img_transform(image_pil).unsqueeze(0)
            
            if self.use_gpu:
                img_tensor = img_tensor.cuda()
                
            with torch.no_grad():
                if self.use_original_model:
                    # Use original AlignedReID feature extraction
                    features_dict = self.feature_extractor(img_tensor)
                    # Extract features from layer '7' and apply pooling
                    if '7' in features_dict:
                        pooled_features = self.pool2d(features_dict['7'], type='max')[0]
                        # Normalize features (same as original)
                        pooled_features = normalize(pooled_features.cpu().numpy())
                        return torch.tensor(pooled_features).squeeze()
                    else:
                        print("⚠️ Layer '7' not found in features")
                        return None
                else:
                    # Use ResNet50 fallback
                    features = self.feature_extractor(img_tensor)
                    features = features.flatten(1)  # Flatten to 1D
                    features = features / features.norm(dim=1, keepdim=True)  # Normalize
                    return features.squeeze(0)  # Remove batch dimension
                
        except Exception as e:
            logger.warning(f"Feature extraction failed: {e}")
            return None

class CLIPPersonIdentifier:
    """CLIP-based person identifier for initial entity recognition"""
    
    def __init__(self, device: str = None, model_name: str = "openai/clip-vit-base-patch32"):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        
        try:
            # Load CLIP model and processor
            self.model = CLIPModel.from_pretrained(model_name).to(self.device)
            self.processor = CLIPProcessor.from_pretrained(model_name)
            self.model.eval()
            
            print(f"✅ CLIP model {model_name} loaded on {self.device}")
            
        except Exception as e:
            print(f"⚠️ CLIP loading failed: {e}")
            self.model = None
            self.processor = None
        
        # Cache for target descriptions
        self.target_embeddings_cache = {}
    
    def add_target_description(self, target_id: int, description: str):
        """Add a target description and cache its embedding"""
        if self.model is None:
            return
            
        try:
            inputs = self.processor(text=[description], return_tensors="pt", padding=True).to(self.device)
            
            with torch.no_grad():
                text_features = self.model.get_text_features(**inputs)
                text_features = text_features / text_features.norm(dim=-1, keepdim=True)
            
            self.target_embeddings_cache[target_id] = {
                'description': description,
                'embedding': text_features.cpu(),
                'matches_found': 0
            }
            
            print(f"Added target {target_id}: '{description}'")
            
        except Exception as e:
            logger.warning(f"Failed to add target description: {e}")
    
    def remove_target_description(self, target_id: int):
        """Remove a target description from cache"""
        if target_id in self.target_embeddings_cache:
            desc = self.target_embeddings_cache[target_id]['description']
            del self.target_embeddings_cache[target_id]
            print(f"Removed target {target_id}: '{desc}'")
    
    def clear_all_targets(self):
        """Clear all target descriptions"""
        count = len(self.target_embeddings_cache)
        self.target_embeddings_cache.clear()
        print(f"Cleared {count} target descriptions")
    
    def identify_new_person(self, frame: np.ndarray, person_bbox, confidence_threshold: float = 0.25):
        """Identify if a new person matches any target descriptions"""
        if self.model is None or not self.target_embeddings_cache:
            return None
        
        try:
            # Extract person crop
            x1, y1, x2, y2 = map(int, person_bbox[:4])
            person_crop = frame[y1:y2, x1:x2]
            
            if person_crop.size == 0:
                return None
            
            # Convert to PIL Image
            person_crop_rgb = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(person_crop_rgb)
            
            # Process image with CLIP
            inputs = self.processor(images=pil_image, return_tensors="pt").to(self.device)
            
            with torch.no_grad():
                image_features = self.model.get_image_features(**inputs)
                image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            
            # Compare with all target descriptions
            best_target_id = None
            best_similarity = 0.0
            best_description = ""
            
            for target_id, target_data in self.target_embeddings_cache.items():
                text_features = target_data['embedding'].to(self.device)
                
                # Compute cosine similarity
                similarity = torch.cosine_similarity(image_features, text_features).item()
                
                if similarity > best_similarity and similarity > confidence_threshold:
                    best_similarity = similarity
                    best_target_id = target_id
                    best_description = target_data['description']
            
            if best_target_id is not None:
                self.target_embeddings_cache[best_target_id]['matches_found'] += 1
                return (best_target_id, best_similarity, best_description)
            
            return None
            
        except Exception as e:
            logger.warning(f"CLIP identification failed: {e}")
            return None

# Initialize the classes
ReId = Aligned_Reid_class()
clip_identifier = CLIPPersonIdentifier()

print("✅ ReID and CLIP identifiers initialized")

### Track Result

In [ ]:
class TrackResult:
    """Represents a single track result"""
    camera_id: str
    local_track_id: int
    bbox: List[float]  # [x1, y1, x2, y2]
    confidence: float
    is_new: bool
    reid_features: Optional[torch.Tensor] = None
    global_target_id: Optional[int] = None


## Kalman

In [ ]:
class KalmanTracker:
    """
    Enhanced Kalman Filter tracker similar to ReTrack-VLM approach
    """
    count = 0
    
    def __init__(self, bbox, frame_idx=0):
        self.id = KalmanTracker.count
        KalmanTracker.count += 1
        
        # Initialize Kalman Filter
        if ADVANCED_TRACKING_AVAILABLE:
            self.kf = KalmanFilter(dim_x=7, dim_z=4)
            # State vector: [x, y, s, r, vx, vy, vs] where:
            # x, y: center coordinates
            # s: scale (area)
            # r: aspect ratio (width/height)
            # vx, vy, vs: velocities
            
            # Transition matrix (constant velocity model)
            self.kf.F = np.array([
                [1,0,0,0,1,0,0],
                [0,1,0,0,0,1,0],
                [0,0,1,0,0,0,1],
                [0,0,0,1,0,0,0],
                [0,0,0,0,1,0,0],
                [0,0,0,0,0,1,0],
                [0,0,0,0,0,0,1]
            ])
            
            # Measurement matrix
            self.kf.H = np.array([
                [1,0,0,0,0,0,0],
                [0,1,0,0,0,0,0],
                [0,0,1,0,0,0,0],
                [0,0,0,1,0,0,0]
            ])
            
            # Measurement noise
            self.kf.R[2:,2:] *= 10.
            
            # Initial state uncertainty
            self.kf.P[4:,4:] *= 1000.  # High uncertainty for velocities
            self.kf.P *= 10.
            
            # Process noise
            self.kf.Q[-1,-1] *= 0.01
            self.kf.Q[4:,4:] *= 0.01
            
            # Initialize state
            self.kf.x[:4] = self.convert_bbox_to_z(bbox)
        else:
            # Fallback to simple position tracking
            self.position = bbox.copy()
            
        # Tracking state
        self.time_since_update = 0
        self.hits = 0
        self.hit_streak = 0
        self.age = 0
        self.last_frame_idx = frame_idx
        
        # Feature galleries for ReID
        self.reid_features = deque(maxlen=10)  # Increased gallery size
        self.clip_features = deque(maxlen=5)
        
        # ReID feature quality tracking
        self.reid_quality_scores = deque(maxlen=10)
        self.avg_reid_quality = 0.0
        
        # Association history for learning
        self.association_history = deque(maxlen=20)
        self.successful_associations = 0
        self.failed_associations = 0
        
        # Motion prediction
        self.velocity_history = deque(maxlen=10)
        self.trajectory = deque(maxlen=20)
        
        # Track quality metrics
        self.confidence_scores = deque(maxlen=10)
        self.avg_confidence = 1.0
        
        # State management
        self.state = 'ACTIVE'  # ACTIVE, LOST, RECOVERING
        self.lost_frames = 0
        self.max_lost_frames = 30
        
    def predict(self):
        """Predict next state using Kalman filter"""
        if ADVANCED_TRACKING_AVAILABLE:
            # Handle negative scale
            if (self.kf.x[6] + self.kf.x[2]) <= 0:
                self.kf.x[6] *= 0.0
            
            self.kf.predict()
        
        self.age += 1
        self.time_since_update += 1
        
        if self.time_since_update > 0:
            self.hit_streak = 0
            
        # Update motion history
        current_bbox = self.get_state()
        if len(current_bbox) > 0:
            cx, cy, w, h = self._bbox_to_center_size(current_bbox[0])
            self.trajectory.append((cx, cy, w, h))
            
            # Calculate velocity
            if len(self.trajectory) >= 2:
                prev_pos = self.trajectory[-2]
                curr_pos = self.trajectory[-1]
                vx = curr_pos[0] - prev_pos[0]
                vy = curr_pos[1] - prev_pos[1]
                self.velocity_history.append((vx, vy))
        
        return self.get_state()
    
    def update(self, bbox, reid_feature=None, clip_feature=None, confidence=1.0, reid_quality=1.0):
        """Update tracker with new detection and enhanced feature management"""
        self.time_since_update = 0
        self.hits += 1
        self.hit_streak += 1
        self.lost_frames = 0
        
        if ADVANCED_TRACKING_AVAILABLE:
            self.kf.update(self.convert_bbox_to_z(bbox))
        else:
            self.position = bbox.copy()
            
        # Enhanced feature update with quality gating
        if reid_feature is not None and reid_quality > 0.5:  # Only update with good quality features
            self.reid_features.append(reid_feature)
            self.reid_quality_scores.append(reid_quality)
            
            # Update average quality
            if self.reid_quality_scores:
                self.avg_reid_quality = np.mean(self.reid_quality_scores)
                
        if clip_feature is not None:
            self.clip_features.append(clip_feature)
            
        # Update confidence
        self.confidence_scores.append(confidence)
        if self.confidence_scores:
            self.avg_confidence = np.mean(self.confidence_scores)
            
        # Update state
        if self.state == 'LOST':
            self.state = 'RECOVERING'
        elif self.state == 'RECOVERING' and self.hit_streak >= 3:
            self.state = 'ACTIVE'
            
        # Record successful association
        self.successful_associations += 1
        self.association_history.append(('success', confidence, reid_quality))
    
    def get_reid_confidence(self):
        """Get confidence in ReID features for this track"""
        if len(self.reid_features) == 0:
            return 0.0
        
        # Consider both feature count and quality
        feature_count_factor = min(1.0, len(self.reid_features) / 5.0)  # Normalize to [0,1]
        quality_factor = self.avg_reid_quality
        
        return feature_count_factor * quality_factor
    
    def should_use_reid_priority(self):
        """Determine if this track should prioritize ReID over IoU"""
        # Use ReID priority for tracks with good feature history
        return (self.get_reid_confidence() > 0.6 and 
                len(self.reid_features) >= 3 and
                self.avg_reid_quality > 0.7)
    
    def mark_missed(self):
        """Mark tracker as missed in current frame"""
        self.lost_frames += 1
        if self.lost_frames > self.max_lost_frames:
            self.state = 'LOST'
    
    def get_state(self):
        """Get current bounding box"""
        if ADVANCED_TRACKING_AVAILABLE:
            return self.convert_x_to_bbox(self.kf.x)
        else:
            return self.position.reshape((1, 4))
    
    def get_predicted_position(self, frames_ahead=1):
        """Get predicted position for future frames"""
        if not self.velocity_history:
            return self.get_state()[0]
            
        current_bbox = self.get_state()[0]
        cx, cy, w, h = self._bbox_to_center_size(current_bbox)
        
        # Average velocity
        if self.velocity_history:
            avg_vx = np.mean([v[0] for v in self.velocity_history])
            avg_vy = np.mean([v[1] for v in self.velocity_history])
            
            # Predict position
            pred_cx = cx + avg_vx * frames_ahead
            pred_cy = cy + avg_vy * frames_ahead
            
            # Convert back to bbox
            x1 = pred_cx - w/2
            y1 = pred_cy - h/2
            x2 = pred_cx + w/2
            y2 = pred_cy + h/2
            
            return np.array([x1, y1, x2, y2])
        
        return current_bbox
    
    def is_active(self):
        return self.state in ['ACTIVE', 'RECOVERING']
    
    def should_delete(self):
        return self.state == 'LOST' and self.lost_frames > self.max_lost_frames * 2
    
    @staticmethod
    def convert_bbox_to_z(bbox):
        """Convert bbox to measurement vector for Kalman filter"""
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        x = bbox[0] + w/2.
        y = bbox[1] + h/2.
        s = w * h  # scale is area
        r = w / float(h) if h != 0 else 1  # aspect ratio
        return np.array([x, y, s, r]).reshape((4, 1))
    
    @staticmethod
    def convert_x_to_bbox(x):
        """Convert state vector to bbox"""
        w = np.sqrt(abs(x[2] * x[3]))
        h = abs(x[2]) / w if w > 1e-6 else 0
        return np.array([
            x[0] - w/2., x[1] - h/2., 
            x[0] + w/2., x[1] + h/2.
        ]).reshape((1, 4))
    
    @staticmethod
    def _bbox_to_center_size(bbox):
        """Convert bbox to center and size"""
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        cx = bbox[0] + w/2
        cy = bbox[1] + h/2
        return cx, cy, w, h

### MISC FUnctions

In [ ]:
def compute_iou_matrix(detections, trackers):
    """Compute IoU matrix between detections and trackers"""
    if len(detections) == 0 or len(trackers) == 0:
        return np.empty((0, 0))
    
    iou_matrix = np.zeros((len(detections), len(trackers)))
    
    for d, det in enumerate(detections):
        for t, trk in enumerate(trackers):
            trk_bbox = trk.get_state()[0]
            iou_matrix[d, t] = compute_iou(det[:4], trk_bbox)
    
    return iou_matrix

def compute_reid_similarity_matrix(reid_features, trackers):
    """Compute ReID similarity matrix between detections and trackers"""
    if len(reid_features) == 0 or len(trackers) == 0:
        return np.empty((0, 0))
    
    similarity_matrix = np.zeros((len(reid_features), len(trackers)))
    
    for d, det_feature in enumerate(reid_features):
        if det_feature is None:
            continue
            
        for t, tracker in enumerate(trackers):
            if len(tracker.reid_features) == 0:
                similarity_matrix[d, t] = 0.0
                continue
                
            # Compute average similarity with feature gallery
            similarities = []
            for track_feature in tracker.reid_features:
                if track_feature is not None and det_feature is not None:
                    try:
                        # Ensure tensors are on the same device
                        if det_feature.device != track_feature.device:
                            track_feature = track_feature.to(det_feature.device)
                        
                        similarity = torch.cosine_similarity(
                            det_feature.unsqueeze(0),
                            track_feature.unsqueeze(0)
                        ).item()
                        similarities.append(similarity)
                    except Exception as e:
                        logger.warning(f"ReID similarity computation failed: {e}")
                        continue
            
            if similarities:
                # Use maximum similarity from gallery for robustness
                similarity_matrix[d, t] = max(similarities)
            else:
                similarity_matrix[d, t] = 0.0
    
    return similarity_matrix

def compute_combined_cost_matrix(detections, trackers, reid_features, crowd_density='medium'):
    """
    Compute combined cost matrix using IoU and ReID features
    Adaptively weights based on crowd density
    """
    if len(detections) == 0 or len(trackers) == 0:
        empty_matrix = np.empty((0, 0))
        return empty_matrix, empty_matrix, empty_matrix
    
    # Compute individual matrices
    iou_matrix = compute_iou_matrix(detections, trackers)
    reid_matrix = compute_reid_similarity_matrix(reid_features, trackers)
    
    # Adaptive weighting based on crowd density
    if crowd_density == 'low':
        iou_weight = 0.7
        reid_weight = 0.3
    elif crowd_density == 'medium':
        iou_weight = 0.5
        reid_weight = 0.5
    else:  # high crowd density
        iou_weight = 0.3
        reid_weight = 0.7
    
    # Convert to cost matrices (1 - score for minimization)
    iou_cost = 1 - iou_matrix
    reid_cost = 1 - reid_matrix
    
    # Combined cost matrix
    combined_cost = iou_weight * iou_cost + reid_weight * reid_cost
    
    return combined_cost, iou_matrix, reid_matrix


def compute_iou(bbox1, bbox2):
    """Compute IoU between two bounding boxes"""
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])
    
    if x2 <= x1 or y2 <= y1:
        return 0.0
    
    intersection = (x2 - x1) * (y2 - y1)
    area1 = (bbox1[2] - bbox1[0]) * (bbox1[3] - bbox1[1])
    area2 = (bbox2[2] - bbox2[0]) * (bbox2[3] - bbox2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0


def associate_detections_to_trackers(detections, trackers, reid_features=None, 
                                    iou_threshold=0.3, reid_threshold=0.6, crowd_density='medium'):
    """
    Enhanced Hungarian algorithm-based association with ReID integration
    """
    if len(trackers) == 0:
        return np.empty((0, 2), dtype=int), np.arange(len(detections)), np.empty((0,), dtype=int)
    
    if len(detections) == 0:
        return np.empty((0, 2), dtype=int), np.empty((0,), dtype=int), np.arange(len(trackers))
    
    # Compute combined cost matrix with adaptive weighting
    combined_cost, iou_matrix, reid_matrix = compute_combined_cost_matrix(
        detections, trackers, reid_features, crowd_density
    )
    
    # Set high cost for associations that fail both IoU and ReID thresholds
    for d in range(len(detections)):
        for t in range(len(trackers)):
            iou_score = iou_matrix[d, t] if len(iou_matrix) > 0 else 0
            reid_score = reid_matrix[d, t] if len(reid_matrix) > 0 else 0
            
            # Association is valid if either IoU OR ReID passes threshold
            # This allows ReID to rescue low IoU associations in crowds
            iou_valid = iou_score >= iou_threshold
            reid_valid = reid_score >= reid_threshold
            
            if not (iou_valid or reid_valid):
                combined_cost[d, t] = 1e5  # Set very high cost
    
    # Hungarian algorithm for optimal assignment
    if ADVANCED_TRACKING_AVAILABLE:
        matched_indices = linear_sum_assignment(combined_cost)
        matched_indices = np.array(list(zip(matched_indices[0], matched_indices[1])))
    else:
        # Fallback to greedy matching with combined scores
        matched_indices = []
        used_detections = set()
        used_trackers = set()
        
        # Create score matrix (higher is better)
        score_matrix = 1 - combined_cost
        
        # Sort by combined score descending
        all_matches = []
        for d in range(len(detections)):
            for t in range(len(trackers)):
                if combined_cost[d, t] < 1e5:  # Only consider valid associations
                    all_matches.append((d, t, score_matrix[d, t]))
        
        all_matches.sort(key=lambda x: x[2], reverse=True)
        
        for d, t, score in all_matches:
            if d not in used_detections and t not in used_trackers:
                matched_indices.append([d, t])
                used_detections.add(d)
                used_trackers.add(t)
        
        matched_indices = np.array(matched_indices) if matched_indices else np.empty((0, 2), dtype=int)
    
    # Filter matches by final validation
    if len(matched_indices) > 0:
        good_matches = []
        for match in matched_indices:
            d, t = match[0], match[1]
            if combined_cost[d, t] < 1e5:  # Valid association
                good_matches.append(match)
        matched_indices = np.array(good_matches) if good_matches else np.empty((0, 2), dtype=int)
    
    # Get unmatched detections and trackers
    matched_det_indices = matched_indices[:, 0] if len(matched_indices) > 0 else np.empty((0,), dtype=int)
    matched_trk_indices = matched_indices[:, 1] if len(matched_indices) > 0 else np.empty((0,), dtype=int)
    
    unmatched_detections = np.array([d for d in range(len(detections)) if d not in matched_det_indices])
    unmatched_trackers = np.array([t for t in range(len(trackers)) if t not in matched_trk_indices])
    
    return matched_indices, unmatched_detections, unmatched_trackers


### DeepSort style Tracker

In [ ]:
class EnhancedDeepSortTracker:
    """
    Enhanced Deep SORT-style tracker with ReID-integrated association
    """
    def __init__(self, camera_id: str):
        self.camera_id = camera_id
        self.trackers = []
        self.frame_count = 0
        self.max_age = 30
        self.min_hits = 3
        self.iou_threshold = 0.3
        self.reid_threshold = 0.6
        
        # Crowd density estimation
        self.recent_detection_counts = deque(maxlen=30)  # Last 30 frames
        self.crowd_density = 'medium'
        
    def estimate_crowd_density(self, num_detections):
        """Estimate crowd density based on recent detection patterns"""
        self.recent_detection_counts.append(num_detections)
        
        if len(self.recent_detection_counts) < 10:
            return self.crowd_density
        
        avg_detections = np.mean(self.recent_detection_counts)
        
        # Dynamic thresholds based on camera setup
        if avg_detections <= 3:
            self.crowd_density = 'low'
        elif avg_detections <= 8:
            self.crowd_density = 'medium'
        else:
            self.crowd_density = 'high'
            
        return self.crowd_density
    
    def compute_reid_quality(self, reid_feature, bbox_area):
        """Estimate quality of extracted ReID feature"""
        if reid_feature is None:
            return 0.0
        
        # Feature norm as quality indicator
        feature_norm = torch.norm(reid_feature).item()
        
        # Bbox area as quality factor (larger area = better feature)
        area_factor = min(1.0, bbox_area / 10000.0)  # Normalize
        
        # Combine factors
        quality = min(1.0, feature_norm * area_factor)
        
        return quality
        
    def update(self, detections: np.ndarray, reid_features: List[torch.Tensor]) -> List[TrackResult]:
        """Enhanced update with ReID-integrated association"""
        self.frame_count += 1
        
        # Estimate crowd density
        crowd_density = self.estimate_crowd_density(len(detections))
        
        # Predict existing trackers
        for tracker in self.trackers:
            tracker.predict()
        
        # Enhanced association with ReID integration
        matched, unmatched_dets, unmatched_trks = associate_detections_to_trackers(
            detections, self.trackers, reid_features, 
            self.iou_threshold, self.reid_threshold, crowd_density
        )
        
        # Update matched trackers with enhanced feature management
        for match in matched:
            det_idx, trk_idx = match[0], match[1]
            detection = detections[det_idx]
            reid_feature = reid_features[det_idx] if det_idx < len(reid_features) else None
            confidence = detection[4] if len(detection) > 4 else 1.0
            
            # Compute ReID quality
            bbox_area = (detection[2] - detection[0]) * (detection[3] - detection[1])
            reid_quality = self.compute_reid_quality(reid_feature, bbox_area)
            
            self.trackers[trk_idx].update(
                detection[:4], 
                reid_feature=reid_feature, 
                confidence=confidence,
                reid_quality=reid_quality
            )
        
        # Mark unmatched trackers as missed
        for trk_idx in unmatched_trks:
            self.trackers[trk_idx].mark_missed()
        
        # Create new trackers for unmatched detections
        for det_idx in unmatched_dets:
            detection = detections[det_idx]
            reid_feature = reid_features[det_idx] if det_idx < len(reid_features) else None
            
            new_tracker = KalmanTracker(detection[:4], self.frame_count)
            if reid_feature is not None:
                bbox_area = (detection[2] - detection[0]) * (detection[3] - detection[1])
                reid_quality = self.compute_reid_quality(reid_feature, bbox_area)
                new_tracker.reid_features.append(reid_feature)
                new_tracker.reid_quality_scores.append(reid_quality)
                new_tracker.avg_reid_quality = reid_quality
            
            self.trackers.append(new_tracker)
        
        # Remove dead trackers
        self.trackers = [t for t in self.trackers if not t.should_delete()]
        
        # Convert to TrackResult format with ReID confidence
        results = []
        for tracker in self.trackers:
            if tracker.hits >= self.min_hits or tracker.time_since_update <= 1:
                bbox = tracker.get_state()[0]
                
                result = TrackResult(
                    camera_id=self.camera_id,
                    local_track_id=tracker.id,
                    bbox=bbox.tolist(),
                    confidence=tracker.avg_confidence,
                    is_new=tracker.hits == 1,
                    reid_features=tracker.reid_features[-1] if tracker.reid_features else None
                )
                results.append(result)
        
        return results
        
class PerCameraTracker:
    """Per-camera tracker using Deep SORT or ByteTrack"""
    
    def __init__(self, camera_id: str, tracker_type: str = 'deepsort'):
        self.camera_id = camera_id
        self.tracker_type = tracker_type
        self.local_tracks = {}  # Maps local track IDs to global target IDs
        
        if tracker_type == 'deepsort':
            self.tracker = EnhancedDeepSortTracker(camera_id)
        else:
            # In production, add ByteTrack implementation
            self.tracker = EnhancedDeepSortTracker(camera_id)  # Fallback for now
            
        logger.info(f"Initialized {tracker_type} tracker for camera {camera_id}")
    
    def process_frame(self, frame: np.ndarray, detections: np.ndarray) -> List[TrackResult]:
        """Process frame and return tracking results"""
        try:
            # Extract ReID features for detections
            reid_features = []
            frame_height, frame_width = frame.shape[:2]
            
            for det in detections:
                x1, y1, x2, y2 = map(int, det[:4])
                
                # Ensure coordinates are within frame bounds
                x1 = max(0, min(x1, frame_width - 1))
                y1 = max(0, min(y1, frame_height - 1))
                x2 = max(x1 + 1, min(x2, frame_width))
                y2 = max(y1 + 1, min(y2, frame_height))
                
                person_crop = frame[y1:y2, x1:x2]
                
                if person_crop.size > 0:
                    try:
                        person_crop_rgb = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)
                        pil_image = Image.fromarray(person_crop_rgb)
                        features = ReId.get_features(pil_image)
                        reid_features.append(features)
                    except Exception as crop_error:
                        logger.warning(f"Failed to extract ReID features: {crop_error}")
                        reid_features.append(None)
                else:
                    reid_features.append(None)
            
            # Update tracker
            tracks = self.tracker.update(detections, reid_features)
            
            # Add global target associations
            for track in tracks:
                if track.local_track_id in self.local_tracks:
                    track.global_target_id = self.local_tracks[track.local_track_id]
            
            return tracks
            
        except Exception as e:
            logger.error(f"Error processing frame for camera {self.camera_id}: {e}")
            return []

class GlobalMultiCameraManager:
    """Global manager for multi-camera tracking with CLIP and ReID"""
    
    def __init__(self):
        self.camera_trackers: Dict[str, PerCameraTracker] = {}
        self.global_targets: Dict[int, Dict] = {}
        self.next_target_id = 1
        self.reid_threshold = 0.6
        self.clip_threshold = 0.25
        
        logger.info("Initialized Global Multi-Camera Manager")
    
    def add_camera(self, camera_id: str, tracker_type: str = 'deepsort'):
        """Add a new camera with specified tracker type"""
        self.camera_trackers[camera_id] = PerCameraTracker(camera_id, tracker_type)
        logger.info(f"Added camera {camera_id} with {tracker_type} tracker")
    
    def add_clip_target(self, description: str, target_name: str = None) -> int:
        """Add a new CLIP target description"""
        target_id = self.next_target_id
        self.next_target_id += 1
        
        if not target_name:
            target_name = f"Target_{target_id}"
        
        self.global_targets[target_id] = {
            'name': target_name,
            'clip_description': description,
            'reid_features': None,
            'active_cameras': {},  # camera_id: local_track_id
            'last_seen': None,
            'first_detected': time.time(),
            'confidence': 0.0
        }
        
        # Add to CLIP identifier
        clip_identifier.add_target_description(target_id, description)
        
        logger.info(f"Added CLIP target {target_id}: '{description}'")
        return target_id
    
    def process_all_cameras(self, camera_frames: Dict[str, np.ndarray]) -> Dict[str, Any]:
        """Process all cameras and return comprehensive results"""
        all_camera_results = {}
        
        # Step 1: Process each camera independently
        for camera_id, frame in camera_frames.items():
            if camera_id not in self.camera_trackers:
                logger.warning(f"No tracker found for camera {camera_id}")
                continue
                
            try:
                # Run YOLO detection with error handling
                results = model(frame, verbose=False)  # Disable verbose output
                
                # Check if results are valid and extract detections
                if results is None or len(results) == 0:
                    all_camera_results[camera_id] = []
                    continue
                
                # Extract detections from YOLO results (updated for newer YOLO versions)
                try:
                    # For newer ultralytics YOLO versions
                    if hasattr(results[0], 'boxes') and results[0].boxes is not None:
                        boxes = results[0].boxes
                        if len(boxes) == 0:
                            all_camera_results[camera_id] = []
                            continue
                        
                        # Convert to numpy format: [x1, y1, x2, y2, conf, class]
                        detections = []
                        for i in range(len(boxes)):
                            box = boxes.xyxy[i].cpu().numpy()  # [x1, y1, x2, y2]
                            conf = boxes.conf[i].cpu().numpy()
                            cls = boxes.cls[i].cpu().numpy()
                            detections.append([box[0], box[1], box[2], box[3], conf, cls])
                        detections = np.array(detections)
                    
                    # Fallback for older YOLO versions with pandas
                    elif hasattr(results, 'pandas') and hasattr(results.pandas(), 'xyxy'):
                        detections = results.pandas().xyxy[0].values
                    else:
                        # Alternative extraction method
                        detections = []
                        for result in results:
                            if hasattr(result, 'boxes') and result.boxes is not None:
                                boxes = result.boxes
                                for i in range(len(boxes)):
                                    box = boxes.xyxy[i].cpu().numpy()
                                    conf = boxes.conf[i].cpu().numpy()
                                    cls = boxes.cls[i].cpu().numpy()
                                    detections.append([box[0], box[1], box[2], box[3], conf, cls])
                        detections = np.array(detections) if detections else np.array([]).reshape(0, 6)
                        
                except Exception as detection_error:
                    logger.error(f"Error extracting detections: {detection_error}")
                    all_camera_results[camera_id] = []
                    continue
                
                # Validate detections format
                if len(detections) == 0 or (len(detections.shape) > 1 and detections.shape[1] < 6):
                    all_camera_results[camera_id] = []
                    continue
                
                # Filter for person detections
                person_detections = detections[detections[:, 5] == 0]  # class 0 = person
                
                if len(person_detections) > 0:
                    # Process with camera tracker
                    camera_results = self.camera_trackers[camera_id].process_frame(
                        frame, person_detections
                    )
                    all_camera_results[camera_id] = camera_results
                else:
                    all_camera_results[camera_id] = []
                    
            except Exception as e:
                logger.error(f"Error processing camera {camera_id}: {e}")
                all_camera_results[camera_id] = []
        
        # Step 2: Global association across cameras
        self.associate_across_cameras(all_camera_results, camera_frames)
        
        # Step 3: Build comprehensive response
        return self.build_response(all_camera_results)
    
    def associate_across_cameras(self, all_camera_results: Dict[str, List[TrackResult]], 
                                camera_frames: Dict[str, np.ndarray]):
        """Handle global associations across cameras"""
        
        # Handle new tracks with CLIP identification
        for camera_id, camera_results in all_camera_results.items():
            frame = camera_frames[camera_id]
            
            for track_result in camera_results:
                if track_result.is_new and track_result.global_target_id is None:
                    self.handle_new_track(track_result, frame)
        
        # Handle cross-camera ReID associations
        self.handle_cross_camera_reid(all_camera_results, camera_frames)
        
        # Clean up old targets
        self.cleanup_old_targets()
    
    def handle_new_track(self, track_result: TrackResult, frame: np.ndarray):
        """Handle new track detection with CLIP identification"""
        try:
            # Try CLIP identification
            clip_match = clip_identifier.identify_new_person(
                frame, track_result.bbox, confidence_threshold=self.clip_threshold
            )
            
            if clip_match:
                target_id, similarity_score, description = clip_match
                
                # Associate local track with global target
                if target_id in self.global_targets:
                    # Update existing target
                    target = self.global_targets[target_id]
                    target['active_cameras'][track_result.camera_id] = track_result.local_track_id
                    target['last_seen'] = time.time()
                    target['confidence'] = max(target['confidence'], similarity_score)
                    
                    # Store ReID features if not already stored
                    if target['reid_features'] is None and track_result.reid_features is not None:
                        target['reid_features'] = track_result.reid_features
                    
                    # Update local association
                    self.camera_trackers[track_result.camera_id].local_tracks[track_result.local_track_id] = target_id
                    track_result.global_target_id = target_id
                    
                    logger.info(f"CLIP identified track {track_result.local_track_id} in camera {track_result.camera_id} as target {target_id} (score: {similarity_score:.3f})")
                    
        except Exception as e:
            logger.error(f"Error handling new track: {e}")
    
    def handle_cross_camera_reid(self, all_camera_results: Dict[str, List[TrackResult]], 
                                camera_frames: Dict[str, np.ndarray]):
        """Handle cross-camera associations using ReID"""
        
        # Collect unassociated tracks
        unassociated_tracks = []
        
        for camera_id, camera_results in all_camera_results.items():
            for track_result in camera_results:
                if (track_result.global_target_id is None and 
                    track_result.reid_features is not None):
                    unassociated_tracks.append(track_result)
        
        # Try to associate with existing global targets
        for track_result in unassociated_tracks:
            best_target_id = None
            best_reid_score = 0.0
            
            for target_id, target_data in self.global_targets.items():
                if target_data['reid_features'] is not None:
                    try:
                        # Ensure tensors are on the same device
                        track_features = track_result.reid_features
                        target_features = target_data['reid_features']
                        
                        if track_features.device != target_features.device:
                            target_features = target_features.to(track_features.device)
                        
                        reid_score = torch.cosine_similarity(
                            track_features.unsqueeze(0),
                            target_features.unsqueeze(0)
                        ).item()
                        
                        if reid_score > best_reid_score and reid_score > self.reid_threshold:
                            best_reid_score = reid_score
                            best_target_id = target_id
                    except Exception as e:
                        logger.warning(f"Cross-camera ReID comparison failed: {e}")
                        continue
            
            # Associate if good match found
            if best_target_id:
                target = self.global_targets[best_target_id]
                target['active_cameras'][track_result.camera_id] = track_result.local_track_id
                target['last_seen'] = time.time()
                
                # Update local association
                self.camera_trackers[track_result.camera_id].local_tracks[track_result.local_track_id] = best_target_id
                track_result.global_target_id = best_target_id
                
                logger.info(f"Cross-camera ReID: Associated camera {track_result.camera_id} track {track_result.local_track_id} with target {best_target_id} (score: {best_reid_score:.3f})")
    
    def cleanup_old_targets(self):
        """Remove old inactive targets"""
        current_time = time.time()
        timeout = 30.0  # 30 seconds timeout
        
        targets_to_remove = []
        for target_id, target_data in self.global_targets.items():
            if (target_data['last_seen'] and 
                current_time - target_data['last_seen'] > timeout):
                targets_to_remove.append(target_id)
        
        for target_id in targets_to_remove:
            # Remove from CLIP identifier
            clip_identifier.remove_target_description(target_id)
            
            # Remove from global targets
            del self.global_targets[target_id]
            
            # Remove from camera trackers
            for camera_tracker in self.camera_trackers.values():
                tracks_to_remove = [k for k, v in camera_tracker.local_tracks.items() if v == target_id]
                for track_id in tracks_to_remove:
                    del camera_tracker.local_tracks[track_id]
            
            logger.info(f"Removed inactive target {target_id}")
    
    def build_response(self, all_camera_results: Dict[str, List[TrackResult]]) -> Dict[str, Any]:
        """Build comprehensive response with enhanced tracking information"""
        response = {
            'hybrid_multi_camera_tracking': True,
            'timestamp': time.time(),
            'global_targets': [],
            'per_camera_results': {},
            'tracking_analytics': {
                'total_targets': len(self.global_targets),
                'active_cameras': len(all_camera_results),
                'total_tracks': sum(len(results) for results in all_camera_results.values()),
                'crowd_density_analysis': {}
            }
        }
        
        # Add global target information
        for target_id, target_data in self.global_targets.items():
            active_cameras = list(target_data['active_cameras'].keys())
            response['global_targets'].append({
                'target_id': target_id,
                'name': target_data['name'],
                'description': target_data['clip_description'],
                'active_cameras': active_cameras,
                'confidence': target_data['confidence'],
                'last_seen': target_data['last_seen'],
                'tracking_method': 'hybrid_reid_motion_clip'
            })
        
        # Add per-camera results with enhanced analytics
        for camera_id, camera_results in all_camera_results.items():
            # Get crowd density from camera tracker
            crowd_density = 'medium'  # Default
            if camera_id in self.camera_trackers:
                crowd_density = self.camera_trackers[camera_id].tracker.crowd_density
            
            tracks = []
            reid_enhanced_tracks = 0
            
            for track in camera_results:
                track_info = {
                    'local_track_id': track.local_track_id,
                    'bbox': track.bbox,
                    'confidence': track.confidence,
                    'is_new': track.is_new,
                    'global_target_id': track.global_target_id,
                    'tracking_method': 'reid_enhanced_motion' if not track.is_new else 'new_detection'
                }
                
                if track.global_target_id:
                    target_data = self.global_targets[track.global_target_id]
                    track_info['target_name'] = target_data['name']
                    track_info['target_description'] = target_data['clip_description']
                    reid_enhanced_tracks += 1
                
                tracks.append(track_info)
            
            response['per_camera_results'][camera_id] = {
                'tracks': tracks,
                'total_tracks': len(tracks),
                'associated_targets': len([t for t in tracks if t['global_target_id']]),
                'crowd_density': crowd_density,
                'reid_enhanced_tracks': reid_enhanced_tracks
            }
            
            # Add to crowd density analysis
            response['tracking_analytics']['crowd_density_analysis'][camera_id] = {
                'density': crowd_density,
                'track_count': len(tracks),
                'reid_usage': f"{reid_enhanced_tracks}/{len(tracks)}" if tracks else "0/0"
            }
        
        return response

# Global hybrid tracker instance
hybrid_tracker = GlobalMultiCameraManager()

def decode_base64_image(encoded_string: str) -> Optional[np.ndarray]:
    """Decode base64 encoded image with validation"""
    try:
        if not encoded_string or not isinstance(encoded_string, str):
            return None
            
        img_data = base64.b64decode(encoded_string)
        nparr = np.frombuffer(img_data, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        # Validate decoded image
        if img is None or img.size == 0:
            logger.warning("Decoded image is empty or invalid")
            return None
            
        return img
    except Exception as e:
        logger.error(f"Error decoding image: {e}")
        return None

In [ ]:
### Video Processing Functions

def process_video_file(video_path: str, camera_id: str, max_frames: int = None) -> List[np.ndarray]:
    """Process video file and return frames"""
    frames = []
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"❌ Error: Could not open video {video_path}")
        return frames
    
    frame_count = 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    print(f"📹 Processing {camera_id}: {total_frames} frames at {fps:.1f} FPS")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frames.append(frame)
        frame_count += 1
        
        if max_frames and frame_count >= max_frames:
            break
            
        if frame_count % 30 == 0:  # Progress update
            print(f"  📊 Processed {frame_count}/{total_frames if not max_frames else max_frames} frames")
    
    cap.release()
    print(f"✅ Completed {camera_id}: {len(frames)} frames loaded")
    return frames

def process_multiple_videos(video_configs: List[Dict], target_descriptions: List[str], 
                          max_frames: int = None, output_video: bool = True) -> Dict:
    """
    Process multiple videos with CLIP target descriptions
    
    Args:
        video_configs: List of dicts with 'path' and 'camera_id' keys
        target_descriptions: List of text descriptions for target people
        max_frames: Maximum frames to process per video (None for all)
        output_video: Whether to create output video files
        
    Returns:
        Dictionary with tracking results and analytics
    """
    
    # Initialize the global tracker
    global hybrid_tracker
    hybrid_tracker = GlobalMultiCameraManager()
    
    # Add cameras
    for config in video_configs:
        camera_id = config['camera_id']
        hybrid_tracker.add_camera(camera_id, 'deepsort')
        print(f"🎥 Added camera: {camera_id}")
    
    # Add target descriptions
    target_ids = []
    for i, description in enumerate(target_descriptions):
        target_id = hybrid_tracker.add_clip_target(description, f"Person_{i+1}")
        target_ids.append(target_id)
        print(f"🎯 Added target {target_id}: '{description}'")
    
    # Load all video frames
    all_video_frames = {}
    for config in video_configs:
        camera_id = config['camera_id']
        video_path = config['path']
        frames = process_video_file(video_path, camera_id, max_frames)
        all_video_frames[camera_id] = frames
    
    if not all_video_frames:
        print("❌ No video frames loaded")
        return {}\n    
    # Process frames synchronously
    results_timeline = []
    max_frame_count = max(len(frames) for frames in all_video_frames.values())
    
    print(f"\\n🚀 Starting tracking for {max_frame_count} frames across {len(video_configs)} cameras...")
    
    # Video writers for output (if enabled)
    video_writers = {}
    if output_video:
        for camera_id, frames in all_video_frames.items():
            if frames:
                h, w = frames[0].shape[:2]
                fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                output_path = f"{camera_id}_tracked.mp4"
                video_writers[camera_id] = cv2.VideoWriter(output_path, fourcc, 10.0, (w, h))
                print(f"📝 Output video: {output_path}")
    
    # Process each frame
    for frame_idx in range(max_frame_count):
        # Collect current frames from all cameras
        current_camera_frames = {}
        
        for camera_id, frames in all_video_frames.items():
            if frame_idx < len(frames):
                current_camera_frames[camera_id] = frames[frame_idx]
        
        if not current_camera_frames:
            continue
            
        # Process with hybrid tracker
        try:
            frame_results = hybrid_tracker.process_all_cameras(current_camera_frames)
            frame_results['frame_index'] = frame_idx
            results_timeline.append(frame_results)
            
            # Draw results on frames (if creating output video)
            if output_video:
                for camera_id, frame in current_camera_frames.items():
                    if camera_id in video_writers:
                        annotated_frame = draw_tracking_results(frame, frame_results, camera_id)
                        video_writers[camera_id].write(annotated_frame)
            
            # Progress update
            if frame_idx % 30 == 0 or frame_idx == max_frame_count - 1:
                print(f"  🔄 Frame {frame_idx + 1}/{max_frame_count} - "
                      f"Targets: {len(frame_results.get('global_targets', []))}, "
                      f"Tracks: {frame_results.get('tracking_analytics', {}).get('total_tracks', 0)}")
                
        except Exception as e:
            print(f"⚠️ Error processing frame {frame_idx}: {e}")
            continue
    
    # Close video writers
    for writer in video_writers.values():
        writer.release()
    
    # Generate comprehensive analytics
    analytics = generate_tracking_analytics(results_timeline, target_descriptions)
    
    print(f"\\n✅ Processing complete!")
    print(f"📊 Total frames processed: {len(results_timeline)}")
    print(f"🎯 Targets tracked: {len(target_descriptions)}")
    print(f"🎥 Cameras: {len(video_configs)}")
    
    return {
        'results_timeline': results_timeline,
        'analytics': analytics,
        'video_configs': video_configs,
        'target_descriptions': target_descriptions
    }

def draw_tracking_results(frame: np.ndarray, results: Dict, camera_id: str) -> np.ndarray:
    """Draw tracking results on frame"""
    annotated_frame = frame.copy()
    
    if camera_id not in results.get('per_camera_results', {}):
        return annotated_frame
    
    camera_results = results['per_camera_results'][camera_id]
    tracks = camera_results.get('tracks', [])
    
    # Color map for different targets
    colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), 
              (255, 0, 255), (0, 255, 255), (128, 0, 128), (255, 165, 0)]
    
    for track in tracks:
        bbox = track['bbox']
        x1, y1, x2, y2 = map(int, bbox)
        
        # Choose color based on global target ID
        target_id = track.get('global_target_id')
        if target_id is not None:
            color = colors[target_id % len(colors)]
            thickness = 3
            label = f"T{target_id}: {track.get('target_name', 'Unknown')}"
        else:
            color = (128, 128, 128)  # Gray for unidentified
            thickness = 2
            label = f"Track {track['local_track_id']}"
        
        # Draw bounding box
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, thickness)
        
        # Draw label
        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
        cv2.rectangle(annotated_frame, (x1, y1 - label_size[1] - 10), 
                     (x1 + label_size[0], y1), color, -1)
        cv2.putText(annotated_frame, label, (x1, y1 - 5), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Draw confidence
        conf_text = f"{track['confidence']:.2f}"
        cv2.putText(annotated_frame, conf_text, (x2 - 50, y2 - 5), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    # Add camera info
    info_text = f"Camera: {camera_id} | Tracks: {len(tracks)} | Crowd: {camera_results.get('crowd_density', 'N/A')}"
    cv2.putText(annotated_frame, info_text, (10, 30), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    return annotated_frame

def generate_tracking_analytics(results_timeline: List[Dict], target_descriptions: List[str]) -> Dict:
    """Generate comprehensive tracking analytics"""
    
    if not results_timeline:
        return {}
    
    analytics = {
        'total_frames': len(results_timeline),
        'target_analytics': {},
        'camera_analytics': {},
        'global_stats': {
            'max_simultaneous_targets': 0,
            'total_track_switches': 0,
            'average_track_length': 0,
            'detection_accuracy': 0
        }
    }
    
    # Analyze each target
    for i, description in enumerate(target_descriptions):
        target_id = i + 1  # Target IDs start from 1
        
        appearances = 0
        total_confidence = 0
        active_frames = 0
        cameras_seen = set()
        
        for frame_result in results_timeline:
            for target in frame_result.get('global_targets', []):
                if target['target_id'] == target_id:
                    appearances += 1
                    total_confidence += target.get('confidence', 0)
                    active_frames += 1
                    cameras_seen.update(target.get('active_cameras', []))
        
        analytics['target_analytics'][f"target_{target_id}"] = {
            'description': description,
            'total_appearances': appearances,
            'active_frames': active_frames,
            'cameras_seen': list(cameras_seen),
            'average_confidence': total_confidence / max(appearances, 1),
            'detection_rate': active_frames / len(results_timeline)
        }
    
    # Camera-specific analytics
    if results_timeline:
        sample_result = results_timeline[0]
        for camera_id in sample_result.get('per_camera_results', {}).keys():
            
            total_tracks = 0
            total_detections = 0
            crowd_densities = []
            
            for frame_result in results_timeline:
                camera_data = frame_result.get('per_camera_results', {}).get(camera_id, {})
                total_tracks += camera_data.get('total_tracks', 0)
                total_detections += len(camera_data.get('tracks', []))
                
                density = camera_data.get('crowd_density', 'medium')
                crowd_densities.append(density)
            
            analytics['camera_analytics'][camera_id] = {
                'total_tracks': total_tracks,
                'total_detections': total_detections,
                'average_tracks_per_frame': total_tracks / len(results_timeline),
                'dominant_crowd_density': max(set(crowd_densities), key=crowd_densities.count) if crowd_densities else 'medium'
            }
    
    # Global statistics
    max_targets = max(len(frame.get('global_targets', [])) for frame in results_timeline)
    analytics['global_stats']['max_simultaneous_targets'] = max_targets
    
    return analytics

## Example Usage

This section demonstrates how to use the hybrid tracking system with multiple video inputs and text descriptions.

In [ ]:
# File upload functionality for Colab
from google.colab import files
import os

def upload_videos():
    """Upload video files to Colab"""
    print("📁 Upload your video files:")
    uploaded = files.upload()
    
    video_files = []
    for filename in uploaded.keys():
        if filename.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
            video_files.append(filename)
            print(f"✅ Video uploaded: {filename}")
        else:
            print(f"⚠️ Skipped non-video file: {filename}")
    
    return video_files

# Example usage - Upload videos if running in Colab
if 'google.colab' in str(get_ipython()):
    print("🌐 Running in Google Colab")
    video_files = upload_videos()
else:
    print("💻 Running locally - specify video paths manually")
    # Example local video paths
    video_files = [
        "sample_video1.mp4",
        "sample_video2.mp4"
    ]

print(f"📹 Available videos: {video_files}")

In [ ]:
# Configuration for video processing
# Modify these settings based on your requirements

# Video configurations
video_configs = []
for i, video_file in enumerate(video_files[:4]):  # Max 4 videos for demo
    video_configs.append({
        'path': video_file,
        'camera_id': f'Camera_{i+1}'
    })

# Target descriptions - modify these based on who you want to track
target_descriptions = [
    "person wearing red shirt",
    "person with blue jacket", 
    "woman with long hair",
    "man wearing glasses"
]

# Processing settings
MAX_FRAMES = 100  # Limit frames for demo (None for all frames)
CREATE_OUTPUT_VIDEO = True  # Whether to create annotated output videos

print("🎯 Target Descriptions:")
for i, desc in enumerate(target_descriptions):
    print(f"  {i+1}. {desc}")
    
print(f"\\n📹 Video Configurations:")
for config in video_configs:
    print(f"  • {config['camera_id']}: {config['path']}")
    
print(f"\\n⚙️ Settings:")
print(f"  • Max frames: {MAX_FRAMES if MAX_FRAMES else 'All'}")
print(f"  • Output videos: {'Yes' if CREATE_OUTPUT_VIDEO else 'No'}")

In [ ]:
# Main execution - Run the hybrid tracking system
print("🚀 Starting Hybrid Multi-Camera Tracking System...")
print("=" * 60)

if not video_configs:
    print("❌ No video configurations found. Please upload videos first.")
else:
    try:
        # Run the tracking system
        results = process_multiple_videos(
            video_configs=video_configs,
            target_descriptions=target_descriptions,
            max_frames=MAX_FRAMES,
            output_video=CREATE_OUTPUT_VIDEO
        )
        
        print("\\n" + "=" * 60)
        print("📊 TRACKING RESULTS SUMMARY")
        print("=" * 60)
        
        # Display analytics
        analytics = results.get('analytics', {})
        
        print(f"\\n🎯 TARGET PERFORMANCE:")
        for target_key, target_data in analytics.get('target_analytics', {}).items():
            print(f"  • {target_data['description']}")
            print(f"    - Detection rate: {target_data['detection_rate']:.1%}")
            print(f"    - Average confidence: {target_data['average_confidence']:.3f}")
            print(f"    - Cameras seen: {len(target_data['cameras_seen'])}")
        
        print(f"\\n📹 CAMERA PERFORMANCE:")
        for camera_id, camera_data in analytics.get('camera_analytics', {}).items():
            print(f"  • {camera_id}")
            print(f"    - Avg tracks/frame: {camera_data['average_tracks_per_frame']:.1f}")
            print(f"    - Crowd density: {camera_data['dominant_crowd_density']}")
            print(f"    - Total detections: {camera_data['total_detections']}")
        
        global_stats = analytics.get('global_stats', {})
        print(f"\\n🌐 GLOBAL STATISTICS:")
        print(f"  • Max simultaneous targets: {global_stats.get('max_simultaneous_targets', 0)}")
        print(f"  • Total frames processed: {analytics.get('total_frames', 0)}")
        
        # Store results for further analysis
        tracking_results = results
        
        print(f"\\n✅ Processing completed successfully!")
        print(f"💾 Results stored in 'tracking_results' variable")
        
        if CREATE_OUTPUT_VIDEO:
            print(f"\\n🎬 Output videos created:")
            for config in video_configs:
                output_file = f"{config['camera_id']}_tracked.mp4"
                print(f"  • {output_file}")
        
    except Exception as e:
        print(f"❌ Error during processing: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# Visualization and results analysis
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_tracking_results(results):
    """Create visualizations of tracking results"""
    
    if not results or 'results_timeline' not in results:
        print("❌ No results to visualize")
        return
    
    timeline = results['results_timeline']
    analytics = results['analytics']
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Hybrid Tracking System - Results Analysis', fontsize=16)
    
    # 1. Targets detected over time
    frame_indices = [frame['frame_index'] for frame in timeline]
    targets_per_frame = [len(frame.get('global_targets', [])) for frame in timeline]
    
    axes[0, 0].plot(frame_indices, targets_per_frame, 'b-', linewidth=2)
    axes[0, 0].set_title('Targets Detected Over Time')
    axes[0, 0].set_xlabel('Frame')
    axes[0, 0].set_ylabel('Number of Targets')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Detection rates by target
    target_names = []
    detection_rates = []
    
    for target_key, target_data in analytics.get('target_analytics', {}).items():
        target_names.append(target_data['description'][:20] + '...' if len(target_data['description']) > 20 else target_data['description'])
        detection_rates.append(target_data['detection_rate'] * 100)
    
    if target_names:
        axes[0, 1].bar(target_names, detection_rates, color='green', alpha=0.7)
        axes[0, 1].set_title('Detection Rate by Target')
        axes[0, 1].set_ylabel('Detection Rate (%)')
        axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. Tracks per camera over time
    camera_ids = list(results['video_configs'][0].keys()) if results.get('video_configs') else []
    if len(camera_ids) > 1:
        for i, config in enumerate(results.get('video_configs', [])):
            camera_id = config['camera_id']
            camera_tracks = []
            
            for frame in timeline:
                camera_data = frame.get('per_camera_results', {}).get(camera_id, {})
                camera_tracks.append(len(camera_data.get('tracks', [])))
            
            axes[1, 0].plot(frame_indices, camera_tracks, label=camera_id, linewidth=2)
        
        axes[1, 0].set_title('Tracks per Camera Over Time')
        axes[1, 0].set_xlabel('Frame')
        axes[1, 0].set_ylabel('Number of Tracks')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Confidence distribution
    all_confidences = []
    for frame in timeline:
        for camera_results in frame.get('per_camera_results', {}).values():
            for track in camera_results.get('tracks', []):
                all_confidences.append(track.get('confidence', 0))
    
    if all_confidences:
        axes[1, 1].hist(all_confidences, bins=20, color='orange', alpha=0.7, edgecolor='black')
        axes[1, 1].set_title('Confidence Score Distribution')
        axes[1, 1].set_xlabel('Confidence Score')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].axvline(np.mean(all_confidences), color='red', linestyle='--', 
                          label=f'Mean: {np.mean(all_confidences):.3f}')
        axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()
    
    return fig

def download_results(results):
    """Download results and output videos"""
    if 'google.colab' not in str(get_ipython()):
        print("💻 Not running in Colab - files available locally")
        return
    
    try:
        # Download analytics as JSON
        import json
        with open('tracking_analytics.json', 'w') as f:
            json.dump(results.get('analytics', {}), f, indent=2)
        
        print("📥 Downloading results...")
        files.download('tracking_analytics.json')
        
        # Download output videos if they exist
        for config in results.get('video_configs', []):
            output_file = f"{config['camera_id']}_tracked.mp4"
            if os.path.exists(output_file):
                print(f"📥 Downloading {output_file}...")
                files.download(output_file)
        
        print("✅ Download complete!")
        
    except Exception as e:
        print(f"❌ Download failed: {e}")

# Run visualization if results exist
if 'tracking_results' in locals() and tracking_results:
    print("📊 Generating visualizations...")
    fig = visualize_tracking_results(tracking_results)
    
    print("\\n📥 Download results and videos:")
    download_results(tracking_results)
else:
    print("⚠️ No tracking results available. Run the main execution cell first.")

## Usage Instructions

### How to Use This Notebook

1. **Install Dependencies**: Run the first cell to install required packages
2. **Upload Videos**: Use the file upload cell to upload your video files (MP4, AVI, MOV, MKV)
3. **Configure Targets**: Modify the `target_descriptions` list with descriptions of people you want to track
4. **Adjust Settings**: Modify `MAX_FRAMES` and other settings as needed
5. **Run Tracking**: Execute the main tracking cell
6. **View Results**: Visualizations and analytics will be generated
7. **Download**: Download tracked videos and analytics JSON

### Features

- **Multi-Camera Support**: Process multiple video streams simultaneously
- **CLIP Integration**: Natural language descriptions for target identification  
- **ReID Enhancement**: Person re-identification across cameras and time
- **Kalman Filtering**: Advanced motion prediction and tracking
- **Hungarian Algorithm**: Optimal detection-to-track association
- **Crowd Density Adaptation**: Automatic algorithm weighting based on scene complexity
- **Comprehensive Analytics**: Detailed performance metrics and visualizations

### Target Description Examples

- `"person wearing red shirt"`
- `"woman with long blonde hair"`
- `"man in business suit"`
- `"person carrying backpack"`
- `"woman wearing glasses"`
- `"tall person in dark clothing"`

### Output Files

- **Tracked Videos**: `Camera_X_tracked.mp4` with bounding boxes and IDs
- **Analytics JSON**: Comprehensive tracking statistics
- **Visualizations**: Performance plots and charts

### Performance Notes

- Processing time depends on video length and number of cameras
- GPU acceleration recommended for faster processing
- Limit `MAX_FRAMES` for testing with large videos
- Memory usage scales with number of simultaneous tracks